# Transformer Review, Part 2: Generating Text, Model Families, and Beyond Text

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parisar/BME6938-Advanced-Biomedical-AI-Spring-2027/blob/main/lectures/notebooks/week03_transformer_review_part2.ipynb)

*Click the badge to open this notebook in Google Colab, then choose File > Save a copy in Drive so your changes are kept.*

**BME 6938 Advanced Biomedical AI · Week 3 · Reference notebook (ungraded)**

This notebook follows Lecture 3-1 step by step, with the same running example (*"The patient denies chest ..."*) and the
same numbers as the slides.

| Section | Lecture step | What you will see |
|---|---|---|
| 1 | Next-token prediction | GPT-2's probabilities for the next token |
| 2 | The prediction head | From the final vector to a probability over the vocabulary |
| 3 | Generating text | A hand-written generation loop, then greedy versus sampling |
| 4 | The causal mask | The mask and the masked softmax, with the slide numbers |
| 5 | Training in parallel | Shifted targets and one loss per position |
| 6 | The decoder block | Masked self-attention, checked so the future cannot leak |
| 7 | Cross-attention | Queries from the decoder, keys and values from the encoder |
| 8 | Two pretraining objectives | Fill in the blank (BERT) versus next token (GPT-2) |
| 9 | Images as tokens | Cutting an image into ViT patches |
| 10 | The cost of attention | How time grows with sequence length |

**How to run it:** same as Part 1 (Colab: *Runtime > Run all*; HiPerGator: a PyTorch kernel in Jupyter). CPU is enough.
The first run downloads DistilGPT-2 (about 350 MB) and BERT-base (about 440 MB).

## 0. Setup

In [ ]:
import importlib, subprocess, sys
for package in ["torch", "transformers"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.set_printoptions(precision=2, suppress=True)
print("PyTorch", torch.__version__)

## 1. A new task: what comes next?  *(slides: "A New Task: What Comes Next?")*

A language model assigns a probability to **every token in the vocabulary** as the next token. We use **DistilGPT-2**, a
small decoder-only model. It was trained on web text, not clinical notes, so its guesses show what general text looks like.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

GPT_NAME = "distilgpt2"
gpt_tokenizer = AutoTokenizer.from_pretrained(GPT_NAME)
gpt = AutoModelForCausalLM.from_pretrained(GPT_NAME)
gpt.eval()

def next_token_probabilities(text, top_k=5):
    # Return the top_k most likely next tokens and their probabilities.
    ids = gpt_tokenizer(text, return_tensors="pt").input_ids
    with torch.no_grad():
        logits = gpt(ids).logits[0, -1]            # scores for the token after the last one
    probs = torch.softmax(logits, dim=-1)          # one probability per vocabulary entry; they sum to 1
    top = torch.topk(probs, top_k)
    return [(gpt_tokenizer.decode(int(i)), float(p)) for p, i in zip(top.values, top.indices)]

for token, p in next_token_probabilities("The patient denies chest"):
    print(f"{token!r:>14}  {p:.3f}")

## 2. The prediction head  *(slides: "The Prediction Head")*

The last token's final vector (size *d* = 768) goes through a linear layer with one output per vocabulary entry
(the **logits**), then softmax. In GPT-2 this layer shares its weights with the embedding table ("tied" weights).

In [ ]:
ids = gpt_tokenizer("The patient denies chest", return_tensors="pt").input_ids
with torch.no_grad():
    out = gpt(ids, output_hidden_states=True)

final_vector = out.hidden_states[-1][0, -1]          # vector of the last token after the last block
print("final vector:", tuple(final_vector.shape))
print("logits:", tuple(out.logits[0, -1].shape), "= one score per vocabulary entry")
print("prediction head tied to the embedding table:",
      gpt.get_output_embeddings().weight.data_ptr() == gpt.get_input_embeddings().weight.data_ptr())

## 3. Generate one token at a time  *(slides: "Generate One Token at a Time")*

Generation is a loop: predict, append the chosen token, run the model again. Below, the loop is written out by hand with
**greedy** decoding (always take the most likely token).

In [ ]:
def generate_greedy(text, n_new_tokens=8):
    ids = gpt_tokenizer(text, return_tensors="pt").input_ids
    for step in range(n_new_tokens):
        with torch.no_grad():
            logits = gpt(ids).logits[0, -1]
        next_id = logits.argmax()                                  # greedy: most likely token
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)          # append it and repeat
        print(f"step {step + 1}: {gpt_tokenizer.decode(ids[0])!r}")
    return gpt_tokenizer.decode(ids[0])

_ = generate_greedy("The patient denies chest")

**Sampling** draws the next token at random from the probabilities. The **temperature** divides the logits before softmax:
below 1 makes the distribution sharper (closer to greedy), above 1 makes it flatter (more random).

In [ ]:
ids = gpt_tokenizer("The patient denies chest", return_tensors="pt").input_ids
for temperature in [0.3, 1.0, 1.5]:
    torch.manual_seed(1)
    sample = gpt.generate(ids, max_new_tokens=12, do_sample=True, temperature=temperature,
                          pad_token_id=gpt_tokenizer.eos_token_id)
    print(f"temperature {temperature}: {gpt_tokenizer.decode(sample[0])!r}")

## 4. The causal mask  *(slides: "The Causal Mask" and "After Softmax, Future Weights Are Zero")*

During training the whole sentence is fed at once, so a token must not attend to the tokens after it. The **causal mask**
sets every score above the diagonal to −∞ before softmax; since $e^{-\infty} = 0$, those weights become 0.
The score matrix below is the one on the slides.

In [ ]:
words = ["The", "patient", "denies", "chest", "pain"]
S = torch.tensor([[1.2, .4, .3, .2, .5],
                  [ .6, 1.4, .5, .3, .2],
                  [ .3, 1.1, 1.0, .4, .6],
                  [ .2, .5, .9, 1.2, 1.5],
                  [ .1, .4, 1.3, 1.4, 1.0]])

n = len(words)
causal_mask = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)   # True above the diagonal = "blocked"
print("mask (True = blocked):\n", causal_mask.int().numpy())

masked_scores = S.masked_fill(causal_mask, float("-inf"))
weights = torch.softmax(masked_scores, dim=-1)
print("\nweights after softmax (each row sums to 1; future positions are 0):")
for w, row in zip(words, weights):
    print(f"{w:>8s}", row.numpy())

## 5. Every position learns at once  *(slides: "Every Position Learns at Once")*

The training targets are the input shifted by one token. With the causal mask, one forward pass gives a prediction (and a
loss) at **every** position. Hugging Face does the shifting for you when you pass `labels=input_ids`.

In [ ]:
text = "The patient denies chest pain."
ids = gpt_tokenizer(text, return_tensors="pt").input_ids
with torch.no_grad():
    out = gpt(ids, labels=ids)

tokens = [gpt_tokenizer.decode(int(i)) for i in ids[0]]
logits = out.logits[0, :-1]                     # prediction made at positions 1 .. n-1
targets = ids[0, 1:]                            # the token that actually came next
per_position = F.cross_entropy(logits, targets, reduction="none")

for i in range(len(targets)):
    print(f"given {''.join(tokens[:i + 1])!r:<32} predict {tokens[i + 1]!r:<10} loss {per_position[i]:.2f}")
print(f"\naverage loss {per_position.mean():.3f}  (Hugging Face reports {out.loss:.3f})")

## 6. The decoder block  *(slides: "The Decoder Block")*

A decoder block is an encoder block with the causal mask inside self-attention. We reuse the classes from Part 1 (redefined
here so this notebook runs on its own) and check the key property: **changing a later token never changes earlier outputs.**

In [ ]:
class MultiHeadAttention(nn.Module):
    # Multi-head attention (same as Part 1). Pass `memory` for cross-attention, `mask` for causal attention.
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads, self.d_head = n_heads, d_model // n_heads
        self.W_Q, self.W_K, self.W_V, self.W_O = (nn.Linear(d_model, d_model) for _ in range(4))

    def forward(self, x, memory=None, mask=None):
        source = x if memory is None else memory          # self-attention: keys/values from x; cross: from memory
        def split(t):
            return t.view(t.shape[0], self.n_heads, self.d_head).transpose(0, 1)
        Q, K, V = split(self.W_Q(x)), split(self.W_K(source)), split(self.W_V(source))
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        out = (weights @ V).transpose(0, 1).reshape(x.shape[0], -1)
        return self.W_O(out), weights

class DecoderBlock(nn.Module):
    # Decoder-only (GPT-style) block: masked self-attention + feed-forward, each with residual + LayerNorm.
    def __init__(self, d_model=64, n_heads=4, d_ff=256):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads)
        self.norm1, self.norm2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.feed_forward = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))

    def forward(self, x):
        n = x.shape[0]
        mask = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)
        attended, _ = self.attention(x, mask=mask)
        x = self.norm1(x + attended)
        return self.norm2(x + self.feed_forward(x))

block = DecoderBlock().eval()
x = torch.randn(5, 64)
x_changed = x.clone(); x_changed[4] = torch.randn(64)      # change only the LAST token

y, y_changed = block(x), block(x_changed)
print("outputs for tokens 1-4 unchanged:", torch.allclose(y[:4], y_changed[:4], atol=1e-6))
print("output for token 5 changed:     ", not torch.allclose(y[4], y_changed[4]))

## 7. Cross-attention  *(slides: "Cross-Attention")*

In an encoder-decoder model, queries come from the **decoder** tokens and keys and values come from the **encoder** output.
With 3 decoder tokens and 5 encoder tokens, the weight matrix is 3 × 5. No mask is needed, because the whole input is known.

In [ ]:
encoder_tokens = ["lungs", "clear", ",", "no", "effusion"]
decoder_tokens = ["<start>", "normal", "chest"]

encoder_output = torch.randn(5, 64)              # stand-in for the encoder's final vectors
decoder_states = torch.randn(3, 64)              # stand-in for the decoder's vectors so far

cross_attention = MultiHeadAttention(64, n_heads=1)
out, w = cross_attention(decoder_states, memory=encoder_output)
print("output:", tuple(out.shape), " weights (decoder tokens x encoder tokens):", tuple(w[0].shape))

## 8. Two ways to pretrain  *(slides: "Two Ways to Pretrain")*

- **Encoder (BERT):** masked language modeling: hide a token and predict it from both sides.
- **Decoder (GPT-2):** next-token prediction: predict each token from the tokens to its left (section 1).

In [ ]:
from transformers import AutoModelForMaskedLM

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
bert_mlm = AutoModelForMaskedLM.from_pretrained("bert-base-uncased").eval()

text = f"The patient {bert_tokenizer.mask_token} chest pain."
inputs = bert_tokenizer(text, return_tensors="pt")
mask_position = inputs.input_ids[0].tolist().index(bert_tokenizer.mask_token_id)
with torch.no_grad():
    probs = torch.softmax(bert_mlm(**inputs).logits[0, mask_position], dim=-1)

print("BERT fills in the blank in:", text)
top = torch.topk(probs, 5)
for p, i in zip(top.values, top.indices):
    print(f"  {bert_tokenizer.decode(int(i)):>12s}  {p:.3f}")

## 9. Images as tokens: the Vision Transformer  *(slides: "Images as Tokens")*

ViT cuts a 224 × 224 image into 16 × 16-pixel patches (14 × 14 = 196 patches), flattens each patch, projects it to a vector
with a learned linear layer, adds positions and a `[CLS]` token, and feeds the sequence to a standard encoder.
We use a synthetic image so nothing needs to be downloaded.

In [ ]:
# A synthetic 224 x 224 RGB "tissue" image: smooth background plus random round "nuclei".
H = W = 224
yy, xx = np.mgrid[0:H, 0:W]
image = np.stack([0.9 - 0.2 * np.sin(xx / 30), 0.75 + 0.1 * np.cos(yy / 25), 0.85 * np.ones((H, W))], axis=-1)
rng = np.random.default_rng(0)
for cy, cx in rng.integers(10, 214, size=(60, 2)):
    image[(yy - cy) ** 2 + (xx - cx) ** 2 < 36] = [0.45, 0.2, 0.6]
image = torch.tensor(np.clip(image, 0, 1), dtype=torch.float32)   # (224, 224, 3), values in [0, 1]

PATCH = 16
patches = image.unfold(0, PATCH, PATCH).unfold(1, PATCH, PATCH)  # (14, 14, 3, 16, 16)
patches = patches.permute(0, 1, 3, 4, 2).reshape(-1, PATCH * PATCH * 3)
print("patches (tokens):", tuple(patches.shape), "= 196 patches of 16 x 16 x 3 pixel values")

patch_embedding = nn.Linear(PATCH * PATCH * 3, 768)            # learned in a real ViT
tokens = patch_embedding(patches)
cls = torch.zeros(1, 768)                                       # learned [CLS] token in a real ViT
sequence = torch.cat([cls, tokens]) + torch.randn(197, 768) * 0.02   # stand-in for learned position vectors
print("sequence for the encoder:", tuple(sequence.shape), "= [CLS] + 196 patch tokens")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(image); axes[0].set_title("image"); axes[0].axis("off")
axes[1].imshow(image)
for k in range(0, 225, PATCH):
    axes[1].axhline(k - 0.5, color="white", lw=0.6); axes[1].axvline(k - 0.5, color="white", lw=0.6)
axes[1].set_title("cut into 196 patches"); axes[1].axis("off"); plt.tight_layout(); plt.show()

## 10. The cost of attention  *(slides: "The Cost of Attention")*

Every token attends to every token, so the score matrix has n² entries. Doubling the length roughly quadruples the work.
The timings depend on your machine; the trend is what matters.

In [ ]:
lengths = [256, 512, 1024, 2048]
times = []
for n in lengths:
    Q = K = V = torch.randn(1, 8, n, 64)          # (batch, heads, tokens, head size)
    start = time.perf_counter()
    for _ in range(5):
        F.scaled_dot_product_attention(Q, K, V)
    times.append((time.perf_counter() - start) / 5)
    print(f"n = {n:5d}: {n * n:>9,d} scores per head, {times[-1] * 1000:7.1f} ms")

plt.figure(figsize=(5, 3.2))
plt.plot(lengths, times, "o-")
plt.xlabel("sequence length n"); plt.ylabel("seconds per attention call"); plt.title("Attention cost grows like n²")
plt.tight_layout(); plt.show()

## Recap

| Idea | Code in this notebook |
|---|---|
| Next-token probabilities | `gpt(ids).logits[0, -1]` then softmax |
| Generation | `generate_greedy`, `gpt.generate(..., do_sample=True, temperature=...)` |
| Causal mask | `torch.triu(..., diagonal=1)` then `masked_fill(-inf)` |
| Training at every position | `gpt(ids, labels=ids)` and the per-position cross-entropy |
| Decoder block | `DecoderBlock` (future cannot change the past) |
| Cross-attention | `MultiHeadAttention(x, memory=encoder_output)` |
| Images as tokens | `unfold` into 16 × 16 patches, then a linear layer |

> **Try it (optional):** change the prompt in section 1 to a clinical phrase of your own. Change the temperature in section 3.
> In section 9, try `PATCH = 32`: how many tokens does the image become, and how does that change the attention cost in section 10?